In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 500)

In [ ]:
PATH = 'clock_template.xlsx'

# ideas: 
# add clock_in and clock_out time as well in visualisation

show_bars = [
            #'hours_worked',
            'extra_hours',
            #'break_time', 
            ]
show_lines = [
            #'hours_worked',
            'lief',
            #'normal_day',
            #'cumulative_extra_hours'
            ]

clock_df = pd.read_excel(PATH, decimal=',')
clock_df['normal_day'] = 8
clock_df['0'] = 0
clock_df['cumulative_extra_hours'] = 0  # placeholder


for i in range(0, len(clock_df)):
    clock_df.loc[i, 'cumulative_extra_hours'] = clock_df['extra_hours'][:i+1].sum()


fig, ax = plt.subplots(figsize=(20, 5))
clock_df.plot(kind='bar', x='date', y=show_bars, ax=ax)
clock_df.plot(kind='line', x='date', y=show_lines, ax=ax, color=['green', 'red','blue'], linewidth=2) 
plt.xticks(rotation=90)
plt.show()


# # separate graph for extra hours over time
# fig_2, ax2 = plt.subplots(figsize=(20, 3))
# clock_df.plot(kind='line',x='date', y=["cumulative_extra_hours", "0"], ylabel='extra hours', xlabel='date', ax=ax2)
# plt.xticks(rotation=90)
# plt.show()

In [ ]:
# import
clock_df = pd.read_excel(PATH, decimal=',')

# definition of good and bad day
good_day = clock_df['lief'] > 5
bad_day = clock_df['lief'] < 5

# best and worst days
clock_df.sort_values('lief', ascending=False)

# homeoffice good or bad?
clock_df[good_day]['homeoffice'].value_counts()

# amount of hours
clock_df['hours_worked_rounded'] = round(clock_df['hours_worked'],2)
clock_df[bad_day]['hours_worked_rounded'].value_counts()

# good vs bad days count
len(clock_df[bad_day]) / len(clock_df) * 100
len(clock_df[good_day]) / len(clock_df) * 100

# starting and finishing hour
clock_df['clock_out'] = pd.to_datetime(clock_df['clock_out'])
clock_df['clock_in'] = pd.to_datetime(clock_df['clock_in'])

for column in ['clock_in', 'clock_out']:
    clock_df[f'{column}_decimal'] = (
        clock_df[column].dt.hour
        + clock_df[column].dt.minute / 60
        + clock_df[column].dt.second / 3600
    )

clock_df['clock_in_decimal'].mean()
clock_df['clock_out_decimal'].mean()
clock_df[bad_day]['clock_in_decimal'].mean()
clock_df[bad_day]['clock_out_decimal'].mean()
clock_df[good_day]['clock_in_decimal'].mean()
clock_df[good_day]['clock_out_decimal'].mean()

# one hot encoding
clock_df = pd.get_dummies(clock_df, columns=["homeoffice"])
clock_df["weekday"] = pd.to_datetime(clock_df["date"]).dt.day_name()
clock_df = pd.get_dummies(clock_df, columns=["weekday"])

# find correlations
clock_df.rename(columns={'Unnamed: 0' : 'days_count'}, inplace=True)
numeric_df = clock_df.select_dtypes(include="number")
corr_with_lief = numeric_df.corr()["lief"].sort_values(ascending=False)
print(corr_with_lief.dropna())